In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import pandas as pd
from datetime import datetime
import glob
from utils import *

In [2]:
def predict(y, *args):
    x = np.array(args).T
    x0 = np.mean(x, axis=0)
    y0 = np.mean(y, axis=0)

    k = np.linalg.pinv(x - x0) @ (y - y0)
    b = y0 - k @ x0
    y_fit = x @ k + b
    return y_fit

In [3]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/wcs.csv', skipinitialspace=True).dropna()

data_dates = np.array([datetime.fromisoformat(date) for date in df['date']])
data_distances = df['dsun_au'].to_numpy()

In [4]:
folder = '/home/ulyanov/data/solo/phi/flat/fdt/flat_newprefilter/'
flat_folders = sorted(glob.glob(folder + '*'))

In [41]:
stokes = []
distances = []
radii = []
dates = []
continuums = []
temperatures = []
velocities = []
wavelengths = []

for flat_folder in flat_folders:
    for flat_file, ghost_file in zip(glob.glob(flat_folder + '/*flat*.fits'), glob.glob(flat_folder + '/*ghost*.fits')):
        with fits.open(flat_file) as hdul:
            flat = hdul[0].data
            flat_header = hdul[0].header

        with fits.open(ghost_file) as hdul:
            ghost = hdul[0].data

        flat = demodulate(flat)
        ghost = demodulate(ghost)
        #temp = np.nanmean(flat[1:,800:1250,800:1250], axis=(1,2))
        #temp = np.nanmean(ghost[1:,800:1250,800:1250], axis=(1,2))
        temp = np.nanmean(flat[1:,800:1250,800:1250] + ghost[1:,800:1250,800:1250], axis=(1,2))

        date = datetime.fromisoformat(flat_header['DATE-OBS'][:-7].replace('.', '').replace('_', 'T'))
        distance = flat_header['DSUN_AU']
        radius = flat_header['RSUN_ARC'] / 60 / 60
        continuum = flat_header['CONTPOSN']
        temperature = round(flat_header['FGOV1PT1'])
        velocity = flat_header['OBS_VR'] / 1000

        wavelength = flat_header['WAVELN0' + str(flat_header['CONTPOS'])]

        stokes.append(temp)
        distances.append(distance)
        radii.append(radius)
        dates.append(date)
        continuums.append(continuum)
        temperatures.append(temperature)
        velocities.append(velocity)
        wavelengths.append(wavelength)

stokes = np.array(stokes).T
distances = np.array(distances)
radii = np.array(radii)
dates = np.array(dates)
continuums = np.array(continuums)
temperatures = np.array(temperatures)
velocities = np.array(velocities)
wavelengths = np.array(wavelengths)

years = np.array([(date - dates[0]).total_seconds() / 24 / 3600 / 365 for date in dates])
#shifts = velocities / 300000 * 6173
shifts = wavelengths - 6173.341

In [39]:
plt.figure(figsize=(10,7))

for continuum in ['red', 'blue']:
    for temperature in [56,61,66]:
        color = {56: 'tab:red', 61: 'tab:green', 66: 'tab:blue'}[temperature]
        marker = {'red': '<', 'blue': '>'}[continuum]
        mask = (temperatures == temperature) * (continuums == continuum)
        plt.scatter(dates[mask], distances[mask], color=color, marker=marker, label=f'{temperature} ({continuum})', s=100)

plt.plot(data_dates, data_distances, '--', lw=1)


plt.xlabel('Date', size=14)
plt.ylabel('Distance, AU', size=14)

plt.ylim(0.2,1)
plt.grid(True)
plt.legend(loc='best')
plt.tight_layout()

In [59]:
for pol in range(3):
    pol_label = {0:'Q', 1:'U', 2:'V'}[pol]

    plt.figure(figsize=(10,7))
    for continuum in ['red', 'blue']:
        for temperature in [56,61,66]:
            color = {56: 'tab:red', 61: 'tab:green', 66: 'tab:blue'}[temperature]
            marker = {'red': '<', 'blue': '>'}[continuum]
            mask = (temperatures == temperature) * (continuums == continuum)
            plt.scatter(distances[mask], stokes[pol,mask], color=color, marker=marker, label=f'{temperature} ({continuum})', s=100)

    prediction = predict(stokes[pol], distances, years, shifts)
    print(np.sqrt(np.mean((prediction - stokes[pol]) ** 2)))

    plt.scatter(distances, prediction, marker='x', color='black', s=100, label='Prediction')

    for i, date in enumerate(dates):
        plt.annotate(date.strftime('%Y-%m-%d'), xy=(distances[i], stokes[pol,i]))

    plt.title(f'Stokes {pol_label}', size=18)
    plt.xlabel('Distance, AU', size=14)

    plt.xlim(0.2,1)

    plt.grid(True)
    plt.legend(loc='best')
    plt.tight_layout()

0.0001065556699104728
0.00014743361498013754
6.233899959014025e-05


In [13]:
for pol in range(3):
    pol_label = {0:'Q', 1:'U', 2:'V'}[pol]

    plt.figure(figsize=(10,7))
    for continuum in ['red', 'blue']:
        for temperature in [56,61,66]:
            color = {56: 'tab:red', 61: 'tab:green', 66: 'tab:blue'}[temperature]
            marker = {'red': '<', 'blue': '>'}[continuum]
            mask = (temperatures == temperature) * (continuums == continuum)
            plt.scatter(dates[mask], stokes[pol,mask], color=color, marker=marker, label=f'{temperature} ({continuum})', s=100)

    prediction = predict(stokes[pol], distances, years, shifts)
    plt.scatter(dates, prediction, marker='x', color='black', s=100, label='Prediction')

    for i, distance in enumerate(distances):
        plt.annotate(f'{np.round(distance,2)}AU', xy=(dates[i], stokes[pol,i]))

    plt.title(f'Stokes {pol_label}', size=18)
    plt.xlabel('Date, AU', size=14)

    plt.grid(True)
    plt.legend(loc='best')
    plt.tight_layout()